In [ ]:
%%sql -r dataframe_1
SELECT 
  SALES_ID
 ,CUSTOMER_ID
 ,PRODUCT_ID
 ,STORE_ID
 ,ORDER_DATE
 ,TRY_CAST(QUANTITY AS NUMBER) AS QUANTITY  
 ,UNIT_PRICE,DISCOUNT
 ,TOTAL_AMOUNT 
 FROM SALES_DB.RAW.SALES WHERE CUSTOMER_ID IS NULL LIMIT 100; 

## Quarantine Bad Records


In [ ]:
%%sql -r quarantine_create
CREATE TABLE IF NOT EXISTS SALES_DB.RAW.SALES_QUARANTINE AS
SELECT
  *,
  CASE
    WHEN CUSTOMER_ID IS NULL AND (QUANTITY IS NULL OR TRY_CAST(QUANTITY AS NUMBER) IS NULL OR TRY_CAST(QUANTITY AS NUMBER) <= 0)
      THEN 'NULL CUSTOMER_ID and INVALID QUANTITY'
    WHEN CUSTOMER_ID IS NULL
      THEN 'NULL CUSTOMER_ID'
    WHEN STORE_ID IS NULL
        THEN 'NULL STORE ID'
    WHEN PRODUCT_ID IS NULL
         THEN 'NULL PRODUCT ID'
    WHEN QUANTITY IS NULL OR TRY_CAST(QUANTITY AS NUMBER) IS NULL
      THEN 'NON-NUMERIC or NULL QUANTITY'
    WHEN TRY_CAST(QUANTITY AS NUMBER) <= 0
      THEN 'NEGATIVE or ZERO QUANTITY'
    WHEN TRY_CAST(DISCOUNT AS FLOAT) IS NULL
        THEN 'INVALID_DISCOUNT'
    WHEN TRY_CAST(ORDER_DATE AS DATE) IS NULL
        THEN 'INVALID DATE OR NO DATE'
  END AS QUARANTINE_REASON,
  CURRENT_TIMESTAMP() AS QUARANTINED_AT
FROM SALES_DB.RAW.SALES
WHERE CUSTOMER_ID IS NULL
   OR QUANTITY IS NULL
   OR TRY_CAST(QUANTITY AS NUMBER) IS NULL
   OR TRY_CAST(QUANTITY AS NUMBER) <= 0;

In [ ]:
%%sql -r quarantine_summary
SELECT QUARANTINE_REASON, COUNT(*) AS BAD_RECORD_COUNT
FROM SALES_DB.RAW.SALES_QUARANTINE
GROUP BY QUARANTINE_REASON
ORDER BY BAD_RECORD_COUNT DESC;

In [ ]:
%%sql -r quarantine_preview
SELECT * FROM SALES_DB.RAW.SALES_QUARANTINE LIMIT 20;

## Storing Clean Data in Silver Layer


In [ ]:
%%sql -r silver_create
CREATE OR REPLACE TABLE SALES_DB.SILVER.SALES AS
SELECT
  SALES_ID,
  CUSTOMER_ID,
  PRODUCT_ID,
  STORE_ID,
  TRY_CAST(ORDER_DATE AS DATE) AS ORDER_DATE,
  TRY_CAST(QUANTITY AS NUMBER) AS QUANTITY,
  TRY_CAST(UNIT_PRICE AS FLOAT) AS UNIT_PRICE,
  TRY_CAST(DISCOUNT AS FLOAT) AS DISCOUNT,
  TRY_CAST(TOTAL_AMOUNT AS FLOAT) AS TOTAL_AMOUNT
FROM SALES_DB.RAW.SALES
WHERE CUSTOMER_ID IS NOT NULL
  AND QUANTITY IS NOT NULL
  AND PRODUCT_ID IS NOT NULL
  AND STORE_ID IS NOT NULL
  AND TRY_CAST(QUANTITY AS NUMBER) IS NOT NULL
  AND TRY_CAST(QUANTITY AS NUMBER) > 0
  AND TRY_CAST(DISCOUNT AS FLOAT) IS NOT NULL
  AND TRY_CAST(ORDER_DATE AS DATE) IS NOT NULL;

In [ ]:
%%sql -r row_counts
SELECT
  (SELECT COUNT(*) FROM SALES_DB.RAW.SALES) AS RAW_COUNT,
  (SELECT COUNT(*) FROM SALES_DB.RAW.SALES_QUARANTINE) AS QUARANTINED_COUNT,
  (SELECT COUNT(*) FROM SALES_DB.SILVER.SALES) AS SILVER_COUNT;

## Automated Pipeline: Stream + Task Graph
Scheduled every 10 minutes. When new data arrives in RAW.SALES:
1. **Root task** checks the stream for new data
2. **Quarantine task** routes bad records to SALES_QUARANTINE
3. **Silver load task** MERGEs clean records into SILVER.SALES

In [ ]:
%%sql -r setup_streams
ALTER TABLE SALES_DB.RAW.SALES SET CHANGE_TRACKING = TRUE;

CREATE OR REPLACE STREAM SALES_DB.RAW.SALES_STREAM
  ON TABLE SALES_DB.RAW.SALES APPEND_ONLY = TRUE;

CREATE OR REPLACE STREAM SALES_DB.RAW.SALES_STREAM_SILVER
  ON TABLE SALES_DB.RAW.SALES APPEND_ONLY = TRUE;

In [ ]:
%%sql -r create_root_task
CREATE OR REPLACE TASK SALES_DB.RAW.SALES_PIPELINE_ROOT
  WAREHOUSE = COMPUTE_WH
  SCHEDULE = '10 MINUTE'
  -- WHEN SYSTEM$STREAM_HAS_DATA('SALES_DB.RAW.SALES_STREAM')
  AS
    SELECT 1;

In [ ]:
%%sql -r create_quarantine_task
CREATE OR REPLACE TASK SALES_DB.RAW.QUARANTINE_BAD_RECORDS
  WAREHOUSE = COMPUTE_WH
  AFTER SALES_DB.RAW.SALES_PIPELINE_ROOT
  AS
    INSERT INTO SALES_DB.RAW.SALES_QUARANTINE
    SELECT
      SALES_ID, CUSTOMER_ID, PRODUCT_ID, STORE_ID, ORDER_DATE,
      QUANTITY, UNIT_PRICE, DISCOUNT, TOTAL_AMOUNT,
      CASE
        WHEN CUSTOMER_ID IS NULL AND (QUANTITY IS NULL OR TRY_CAST(QUANTITY AS NUMBER) IS NULL OR TRY_CAST(QUANTITY AS NUMBER) <= 0)
          THEN 'NULL CUSTOMER_ID and INVALID QUANTITY'
        WHEN CUSTOMER_ID IS NULL
          THEN 'NULL CUSTOMER_ID'
        WHEN QUANTITY IS NULL OR TRY_CAST(QUANTITY AS NUMBER) IS NULL
          THEN 'NON-NUMERIC or NULL QUANTITY'
        WHEN TRY_CAST(QUANTITY AS NUMBER) <= 0
          THEN 'NEGATIVE or ZERO QUANTITY'
      END AS QUARANTINE_REASON,
      CURRENT_TIMESTAMP() AS QUARANTINED_AT
    FROM SALES_DB.RAW.SALES_STREAM
    WHERE CUSTOMER_ID IS NULL
       OR QUANTITY IS NULL
       OR TRY_CAST(QUANTITY AS NUMBER) IS NULL
       OR TRY_CAST(QUANTITY AS NUMBER) <= 0

In [ ]:
%%sql -r create_silver_task
CREATE OR REPLACE TASK SALES_DB.RAW.LOAD_SILVER_SALES
  WAREHOUSE = COMPUTE_WH
  AFTER SALES_DB.RAW.QUARANTINE_BAD_RECORDS
  AS
    MERGE INTO SALES_DB.SILVER.SALES AS t
    USING (
      SELECT
        SALES_ID,
        CUSTOMER_ID,
        PRODUCT_ID,
        STORE_ID,
        TRY_CAST(ORDER_DATE AS DATE) AS ORDER_DATE,
        TRY_CAST(QUANTITY AS NUMBER) AS QUANTITY,
        TRY_CAST(UNIT_PRICE AS FLOAT) AS UNIT_PRICE,
        TRY_CAST(DISCOUNT AS FLOAT) AS DISCOUNT,
        TRY_CAST(TOTAL_AMOUNT AS FLOAT) AS TOTAL_AMOUNT
      FROM SALES_DB.RAW.SALES_STREAM_SILVER
      WHERE CUSTOMER_ID IS NOT NULL
        AND PRODUCT_ID IS NOT NULL
        AND STORE_ID IS NOT NULL
        AND QUANTITY IS NOT NULL
        AND TRY_CAST(QUANTITY AS NUMBER) IS NOT NULL
        AND TRY_CAST(QUANTITY AS NUMBER) > 0
    ) AS s
    ON t.SALES_ID = s.SALES_ID
    WHEN NOT MATCHED THEN INSERT (SALES_ID, CUSTOMER_ID, PRODUCT_ID, STORE_ID, ORDER_DATE, QUANTITY, UNIT_PRICE, DISCOUNT, TOTAL_AMOUNT)
    VALUES (s.SALES_ID, s.CUSTOMER_ID, s.PRODUCT_ID, s.STORE_ID, s.ORDER_DATE, s.QUANTITY, s.UNIT_PRICE, s.DISCOUNT, s.TOTAL_AMOUNT);

In [ ]:
%%sql -r dataframe_2
ALTER TASK SALES_DB.RAW.LOAD_SILVER_SALES SUSPEND;
ALTER TASK SALES_DB.RAW.QUARANTINE_BAD_RECORDS SUSPEND;
ALTER TASK SALES_DB.RAW.SALES_PIPELINE_ROOT SUSPEND;

In [ ]:
%%sql -r resume_tasks
ALTER TASK SALES_DB.RAW.LOAD_SILVER_SALES RESUME;
ALTER TASK SALES_DB.RAW.QUARANTINE_BAD_RECORDS RESUME;
ALTER TASK SALES_DB.RAW.SALES_PIPELINE_ROOT RESUME;

In [ ]:
%%sql -r verify_tasks
SHOW TASKS IN SCHEMA SALES_DB.RAW;